# Wiener-Filter:

Der Wiener-Filter ist ein verbreiteter Algorithmus zur Unterdrueckung von Rauschen und ein gutes Beispiel fuer die Arbeit mit dem Spektrogramm.

In [15]:
import numpy as np
from tqdm import tqdm
import sys
sys.path.insert(1, '../Basics')
import WaveFileHandling

x, r = WaveFileHandling.ReadWaveAsNumpyArray('../Audio/P501_D_EN_fm_SWB_48k.wav')
TimeMemoryOfInput = 3

## Implementierung von STFT und ISTFT
Im folgenden Codeblock werden die STFT und ihre inverse Transformation, die ISTFT, fuer die weitere Verwendung implementiert.

In [16]:
def GetWindowFunction(ws):
    return 0.5 - 0.5 * np.cos(2*np.pi*(np.arange(ws)+0.5)/ws)

class CSpectrogram(object):

    def __init__(self, r):
        self.__r = r

    def GetHopSizeInMilliseconds(self):
        return 10
    
    def GetHopSizeInSamples(self):
        return int(self.__r * self.GetHopSizeInMilliseconds() / 1000)
    
    def GetWindowSizeInSamples(self):
        OverLappingInPercent = 75
        return int(self.GetHopSizeInSamples() / (1 - OverLappingInPercent/100))
    
    def EvalSpectrogram(self, x):    
        hs = self.GetHopSizeInSamples()
        ws = self.GetWindowSizeInSamples()
        w = GetWindowFunction(ws)
        FFTLen = int(2**np.ceil(np.log2(ws)))
        NumberOfBlocks = int((x.shape[0] - ws) / hs) + 1
        X = np.zeros((FFTLen // 2 + 1, NumberOfBlocks), dtype=complex)
        for BlockNumber in range(NumberOfBlocks):
            idx1 = BlockNumber * hs
            idx2 = idx1 + ws
            BlockAnalysis = x[idx1:idx2] * w
            SpectrumAnalysis = np.fft.rfft(BlockAnalysis, n = FFTLen)
            X[:, BlockNumber] = SpectrumAnalysis
        return X

    def EvalTimeDomainSignal(self, X):
        hs = self.GetHopSizeInSamples()
        ws = self.GetWindowSizeInSamples()
        w = GetWindowFunction(ws)
        x = np.zeros(((X.shape[1] - 1) * hs + ws))
        for BlockNumber in range(X.shape[1]):
            idx1 = BlockNumber * hs
            idx2 = idx1 + ws
            BlockSynthesis = np.fft.irfft(X[:, BlockNumber])
            x[idx1:idx2] += BlockSynthesis[:ws] * w * 2 / 3
        return x

ASpectrogram = CSpectrogram(r)
X = ASpectrogram.EvalSpectrogram(x)
y = ASpectrogram.EvalTimeDomainSignal(X)
hs = ASpectrogram.GetHopSizeInSamples()
ws = ASpectrogram.GetWindowSizeInSamples()
x_tmp = x[:y.shape[0]]
x_tmp = x_tmp[ws:-ws]
y_tmp = y[ws:-ws]
SNR = 10 * np.log10(np.sum(x_tmp**2) / np.sum((x_tmp - y_tmp)**2))
assert SNR > 100, f"SNR is too low: {SNR:.2f} dB"

## Rauschmodell
Das Rauschmodell ist additives weisses gaussverteiltes Rauschen (AWGN):

$y(n)=x(n)+d(n)$

wobei $x(n)$ das originale rauschfreie Signal, $d(n)$ das weisse gaussverteilte Rauschen und $y(n)$ das verrauschte Signal ist.

In [17]:
def ApplyNoise(x):
    return x + np.random.randn(x.shape[0]) * 0.1

## Implementierung
Der Wiener-Filter arbeitet in seiner einfachsten Implementierung im Spektralbereich:

$H(f)=\frac{\left|X(f)\right|^2}{\left|X(f)\right|^2+\left|D(f)\right|^2}$

$Z(f) = Y(f)\cdot H(f)$

$\left|X(f)\right|^2$ ist das sogenannte Leistungsdichtespektrum von $x(n)$.

$\left|D(f)\right|^2$ ist das sogenannte Leistungsdichtespektrum des Rauschsignals $d(n)$.

Zusaetzlich kann angenommen werden, dass das Signal $x(n)$ und das Rauschen $d(n)$ unkorreliert sind. In diesem Fall kann Folgendes gezeigt werden:

$\left|Y(f)\right|^2 \approx \left|X(f)\right|^2 + \left|D(f)\right|^2$.

Diese Aussage wird gezeigt, indem der relative Fehler zwischen dem wahren Wert $\left|Y(f)\right|^2$ und der Schaetzung $\left|X(f)\right|^2 + \left|D(f)\right|^2$ berechnet wird.

In [18]:
r = 48000
t = np.arange(r)/r
f = 440
x = np.sin(2*np.pi*f*t)
d = np.random.randn(x.shape[0])*0.1
y = x + d
w = GetWindowFunction(ws)

P_Y = None
idx1 = 0
idx2 = ws
while idx2 < y.shape[0]:
    # evaluate the spectra
    X = np.fft.fft(x[idx1:idx2] * w)
    D = np.fft.fft(d[idx1:idx2] * w)
    Y = np.fft.fft(y[idx1:idx2] * w)

    P_Y_local = (np.abs(Y)**2).reshape((X.shape[0], 1))
    P_Y_hat_local = (np.abs(X)**2 + np.abs(D)**2).reshape((X.shape[0], 1))
    if P_Y is None:
        P_Y = P_Y_local
        P_Y_hat = P_Y_hat_local
    else:
        P_Y = np.concatenate((P_Y, P_Y_local), axis = 1)
        P_Y_hat = np.concatenate((P_Y_hat, P_Y_hat_local), axis = 1)
    
    idx1 += hs
    idx2 += hs

RelativeSquaredError = np.sum((P_Y - P_Y_hat)**2) / np.sum(P_Y**2)
print('relative error = ', RelativeSquaredError)

relative error =  8.51564087560314e-05


Daraus kann die folgende Naeherung angenommen werden:

$H(f)=\frac{\left|X(f)\right|^2}{\left|X(f)\right|^2+\left|D(f)\right|^2}\approx\frac{\left|X(f)\right|^2}{\left|Y(f)\right|^2}$

Im Folgenden wird der Wiener-Filter berechnet und auf Spektrogramme angewendet. Das [SNR](../Basics/SignalToNoiseRatio.ipynb) vor und nach Anwendung des Wiener-Filters wird berechnet.

In [19]:
class CMeanSNREvaluator(object):

    def __init__(self, FilterEstimator):
        self.__MeanSNRWithoutDenoising = 0.0
        self.__MeanSNRwithDenoising = 0.0
        self.__counter = 0
        self.__FilterEstimator = FilterEstimator

    def ProcessSingleFile(self, Filename):
        x, Fs = WaveFileHandling.ReadWaveAsNumpyArray(Filename)
        y = ApplyNoise(x)
        ASpectrogram = CSpectrogram(Fs)
        X = ASpectrogram.EvalSpectrogram(x)
        Y = ASpectrogram.EvalSpectrogram(y)
        X = np.abs(X)
        Y = np.abs(Y)
        DenoiserEstimation = Y * self.__FilterEstimator(X, Y)

        self.__MeanSNRWithoutDenoising += 10*np.log10(np.sum(X**2) / np.sum((X - Y)**2))
        self.__MeanSNRwithDenoising += 10*np.log10(np.sum(X**2) / np.sum((X - DenoiserEstimation)**2))
        self.__counter += 1

    def PrintMeanSNR(self):
        print('Mean SNR without denoising = ', self.__MeanSNRWithoutDenoising / self.__counter)
        print('Mean SNR with denoising = ', self.GetMeanSNRWithDenoising())

    def GetMeanSNRWithDenoising(self):
        return self.__MeanSNRwithDenoising / self.__counter

def FilterEstimator1(X, Y):
    H = (X**2 + 1e-16) / (Y**2 + 1e-16)
    return H

AMeanSNREvaluator = CMeanSNREvaluator(FilterEstimator1)
AMeanSNREvaluator.ProcessSingleFile('../Audio/P501_D_EN_fm_SWB_48k.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/PferdeSchnaubenNichtDieNase.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/SevenEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/TreeEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/ZeroEnglishFemale.wav')
AMeanSNREvaluator.PrintMeanSNR()

Mean SNR without denoising =  -2.7432034455813223
Mean SNR with denoising =  9.128788444719508


## Rauschschaetzung
Leider sind in einer realen Messumgebung das wahre Signal $x$ und das wahre Rauschen $d(n)$ unbekannt. Stattdessen ist nur die Mischung bekannt:

$y(n)=x(n)+d(n)$

Um das Rauschniveau abzuschaetzen, wird Stationaritaet des Rauschens angenommen.
Das bedeutet, dass die Leistungsspektren des Rauschens $\left|D(f)\right|^2$ ueber die Zeit nahezu konstant sind. Diese Annahme ist gueltig, wenn sich das Rauschen ueber die Zeit nicht stark aendert.
Wenn Stationaritaet des Rauschens angenommen wird, kann das Leistungsspektrum $\left|D(f)\right|^2$ berechnet werden, indem der Mittelwert ueber die Spalten $t$ der Leistungsspektren $\left|Y(f, t)\right|^2$ gebildet wird:

$\left|D(f)\right|^2 \approx \frac{1}{T} \sum_{t=0}^{T-1} \left|Y(f, t)\right|^2$.

Daraus kann der Filter berechnet werden durch:

$H(f)=\frac{\left|X(f)\right|^2}{\left|X(f)\right|^2 + \left|D(f)\right|^2}=\frac{\left|Y(f)\right|^2 - \left|D(f)\right|^2}{\left|Y(f)\right|^2}=1-\frac{\left|D(f)\right|^2}{\left|Y(f)\right|^2}$

Hinweis: Die oben angegebene Gleichung ist nur sinnvoll, wenn $\left|D(f)\right|^2 < \left|Y(f)\right|^2$ fuer alle $f$ gilt.

Deshalb wird der Term $\frac{\left|D(f)\right|}{\left|Y(f)\right|}$ im Folgenden auf den Bereich $0..1$ beschraenkt.

In [20]:
def FilterEstimator2(X, Y):
    D = np.sqrt(np.mean(Y**2, axis = 1))
    D = np.outer(D, np.ones(Y.shape[1]))
    H = 1 - (D**2 + 1e-16) / (Y**2 + 1e-16)
    H = np.maximum(H, 0.0)
    H = np.minimum(H, 1.0)
    return H

AMeanSNREvaluator = CMeanSNREvaluator(FilterEstimator2)
AMeanSNREvaluator.ProcessSingleFile('../Audio/P501_D_EN_fm_SWB_48k.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/PferdeSchnaubenNichtDieNase.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/SevenEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/TreeEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/ZeroEnglishFemale.wav')
AMeanSNREvaluator.PrintMeanSNR()

Mean SNR without denoising =  -2.746548066841041
Mean SNR with denoising =  2.5353229670121977


## Feinabstimmung des Wiener-Filters
Im Buch Digitale Sprachsignalverarbeitung von Prof. Vary et al. wird eine Variante des Wiener-Filters mit einem Abstimmparameter $\alpha$ eingefuehrt:

$H(f)=1-\left(\frac{\left|D(f)\right|^2}{\left|Y(f)\right|^2}\right)^\alpha$

Fuer $\alpha=1$ wird der Wiener-Filter implementiert.

Fuer $\alpha=\frac{1}{2}$ wird das Rauschspektrum subtrahiert.

Im Folgenden wird ein [Random Walk](../Basics/RandomWalk.ipynb) implementiert, der einen optimalen Parameter $\alpha$ sucht, der fuer die gegebenen Trainingsdaten zum hoechstmoeglichen SNR fuehrt.

In [ ]:
def FilterEstimator3(X, Y):
    D = np.sqrt(np.mean(Y**2, axis = 1))
    D = np.outer(D, np.ones(Y.shape[1]))
    H = 1 - ((D**2 + 1e-16) / (Y**2 + 1e-16))**alpha
    H = np.maximum(H, 0.0)
    H = np.minimum(H, 1.0)
    return H

SNR = -np.inf
alpha_opt = 0.464
for n in range(50):
    alpha = alpha_opt
    if n > 0:
        alpha += np.random.randn(1) * 0.1
        alpha = np.abs(alpha[0])
    AMeanSNREvaluator = CMeanSNREvaluator(FilterEstimator3)
    AMeanSNREvaluator.ProcessSingleFile('../Audio/P501_D_EN_fm_SWB_48k.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/PferdeSchnaubenNichtDieNase.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/SevenEnglishFemale.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/TreeEnglishFemale.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/ZeroEnglishFemale.wav')        
    MeanSNRwithDenoising = AMeanSNREvaluator.GetMeanSNRWithDenoising()
    if MeanSNRwithDenoising > SNR:
        SNR = MeanSNRwithDenoising
        alpha_opt = alpha
        print('best SNR = ', SNR)
        print('for alpha = ', alpha_opt)

best SNR =  3.5215531251241456
for alpha =  0.499
best SNR =  3.5284224100420554
for alpha =  0.46432830164433453


## Feinabstimmung der Rauschschaetzung
Bisher wird das Rauschspektrum $\left|D(f)\right|$ als quadratischer Mittelwert der gemessenen verrauschten Spektren geschaetzt:

$\left|D(f)\right| \approx \sqrt{\frac{1}{T} \sum_{t=0}^{T-1} \left|Y(f, t)\right|^2}$

Dies kann zum Hoelder-Mittel der letzten Spektren erweitert werden:

$\|D(f)\|_p=\left(\frac{1}{T}\sum_{t=0}^{T-1} \left|Y(f, t)\right|^p\right)^{(1/p)}$

Der quadratische Mittelwert entspricht $p=2$.

Diese Erweiterung des moeglichen Parameterraums erlaubt dem [Random Walk](../Basics/RandomWalk.ipynb) eine breitere Suche nach den besten Parametern, die bezueglich der gegebenen Trainingsdaten zum hoechstmoeglichen SNR fuehren:

In [22]:
from scipy.stats import pmean

def FilterEstimator4(X, Y):
    D = pmean(Y, p=p, axis = 1)
    D = np.outer(D, np.ones(Y.shape[1]))
    H = 1 - ((D**2 + 1e-16) / (Y**2 + 1e-16))**alpha
    H = np.maximum(H, 0.0)
    H = np.minimum(H, 1.0)
    return H

SNR = -np.inf
alpha_opt = 0.283
p_opt = -0.359
for n in range(50):
    alpha = alpha_opt
    p = p_opt
    if n > 0:
        alpha += np.random.randn(1)*0.1
        alpha = np.abs(alpha[0])
        p += np.random.randn(1)
        p = p[0]
    AMeanSNREvaluator = CMeanSNREvaluator(FilterEstimator4)
    AMeanSNREvaluator.ProcessSingleFile('../Audio/P501_D_EN_fm_SWB_48k.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/PferdeSchnaubenNichtDieNase.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/SevenEnglishFemale.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/TreeEnglishFemale.wav')
    AMeanSNREvaluator.ProcessSingleFile('../Audio/ZeroEnglishFemale.wav')        
    MeanSNRwithDenoising = AMeanSNREvaluator.GetMeanSNRWithDenoising()
    if MeanSNRwithDenoising > SNR:
        SNR = MeanSNRwithDenoising
        alpha_opt = alpha
        p_opt = p
        print('best SNR = ', SNR)
        print('for alpha = ', alpha_opt)
        print('and p = ', p)

best SNR =  4.797613042265072
for alpha =  0.283
and p =  -0.359


## Programmieraufgabe:

Bisher ist der Wiener-Filter nicht-kausal implementiert: $H(f)$ wird ueber das gesamte Spektrogramm $Y(f,t)$ berechnet, also auch fuer Spalten $t$, die in der Zukunft liegen.
Implementieren Sie die kausale Prozedur EvaluateCausalDenoiserEstimation, indem Sie nur Spalten von $Y(f,t)$ analysieren, die in der Vergangenheit liegen.

In [ ]:
def FilterEstimator5(X, Y):    
    H = np.zeros(Y.shape)
    ### solution begins

    ### solution ends
    return H

X = np.random.randn(500, 100)
Y = np.random.randn(X.shape[0], Y.shape[0])
H1 = FilterEstimator5(X, Y)
RandomColumn = np.random.randint(0, Y.shape[1])
H2 = FilterEstimator2(X, Y[:, :RandomColumn+1])
assert np.allclose(H1[:, RandomColumn], H2[:, RandomColumn]), f"FilterEstimator5 is not correct for column {RandomColumn}"

AMeanSNREvaluator = CMeanSNREvaluator(FilterEstimator5)
AMeanSNREvaluator.ProcessSingleFile('../Audio/P501_D_EN_fm_SWB_48k.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/PferdeSchnaubenNichtDieNase.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/SevenEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/TreeEnglishFemale.wav')
AMeanSNREvaluator.ProcessSingleFile('../Audio/ZeroEnglishFemale.wav')
AMeanSNREvaluator.PrintMeanSNR()


Mean SNR without denoising =  -2.7281961059449467
Mean SNR with denoising =  2.3043812616175483


## Klausurvorbereitung
1) Ist der vorgeschlagene Wiener-Filter $H(f)=\frac{\left|X(f)\right|^2}{\left|X(f)\right|^2 + \left|D(f)\right|^2}$ komplexwertig?
2) Bestimmen Sie den moeglichen Wertebereich von $H(f)$.
3) Skizzieren Sie den Wiener-Filter $H(f)$ fuer weisses Rauschen $\left|D(f)\right|^2=1$, $0<f<10$ und $X(f)=f$.
4) Skizzieren Sie das SNR in Abhaengigkeit von $f$ fuer die gegebenen Parameter aus Aufgabe 3).
5) Bei der Berechnung des Wiener-Filters wird eine kleine positive Zahl $\varepsilon=10^-16$ zum Nenner und zum Zaehler addiert. Warum?

## Zusammenfassung
Nach der Arbeit mit diesem Jupyter-Notebook sollten Sie die folgenden Themen erklaeren koennen:

- Was ist die Motivation fuer die Verwendung des Wiener-Filters?
- Welches Rauschmodell bzw. Kanalmodell wird fuer den Wiener-Filter verwendet?
- Welcher Zusammenhang besteht zwischen dem Leistungsdichtespektrum des Rauschens und dem Originalsignal?
- Welche Varianten zur Schaetzung des unbekannten Rauschens kennen Sie?